# 03. 시군구별 특징 벡터 생성 + 3대 세그먼트 분류 (Step 2~3)

이 노트북이 본 프로젝트의 핵심이다.
(A) 규칙기반 스코어링과 (B) K-means 군집분석으로 각 시군구를 분류하고,
(C) 두 결과를 교차검증한 뒤 최종 세그먼트 라벨(생활밀착형/로컬미식형/프리미엄외식형)을 부여한다.

⚠️ §3-2의 예시 지역 리스트는 사전탐색 단계의 참고용일 뿐이며, 이 노트북은
233개(복합키 기준 255개) 시군구 **전수**를 재현 가능한 알고리즘으로 분류한다.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from feature_engineering import build_region_features, BUZ_LIST, AGE_LABEL_LIST, REGION_KEYS
from scoring import compute_rule_scores, run_kmeans_clustering, map_clusters_to_segments, classify_segments, SEG_A, SEG_B, SEG_C, RANDOM_STATE

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)


## 3-1. 시군구별 특징 벡터 생성 (Step 2)

업종별/연령별 금액 비중, 총소비금액/건수, 건당 평균결제금액, 월별 변동계수(CV), low_sample 플래그.

In [2]:
foreign_df = pd.read_csv('../data/processed/foreign_consumption_clean.csv', dtype={'STRD_YYMM': str})
features = build_region_features(foreign_df)

print(f"지역 수: {len(features)}개 (SIDO_NM+CCG_NM 복합키 기준)")
print(f"low_sample 임계치(하위 10분위 총건수): {features.attrs['low_sample_threshold']:.0f}건")
print(f"low_sample=True 지역 수: {features['low_sample'].sum()}개")
features[REGION_KEYS + ['총소비금액', '총건수', '건당평균결제금액', '월별변동계수CV', 'low_sample']].describe()


지역 수: 255개 (SIDO_NM+CCG_NM 복합키 기준)
low_sample 임계치(하위 10분위 총건수): 22534건
low_sample=True 지역 수: 26개


,총소비금액,총건수,건당평균결제금액,월별변동계수CV
count,2.550000e+02,2.550000e+02,255.000000,255.000000
mean,5.012116e+09,2.961166e+05,17698.536710,0.092178
std,8.953805e+09,4.707158e+05,3121.659413,0.069072
min,8.839000e+07,4.315000e+03,10296.345613,0.021795
25%,1.215220e+09,6.450200e+04,15660.087877,0.052093
50%,3.156210e+09,1.805330e+05,17276.686372,0.067719
75%,6.062445e+09,3.684670e+05,19705.794226,0.108032
max,1.192394e+11,5.938783e+06,29187.857457,0.570328


## 3-2. (A) 규칙기반 스코어링

- `생활밀착지수` = (슈퍼마켓+편의점+대형할인점 비중) − (외식 5종: 서양음식·일반한식·일식회집·한정식·갈비전문점 비중 합)
  (중국음식·제과점·스넥은 두 성격 어디에도 속하지 않는 중간적 업종이라 제외 — 자세한 근거는 `src/scoring.py` 주석 참조)
- `한식지향지수` = 일반한식+한정식+갈비전문점
- `양식지향지수` = 서양음식+일식회집
- 1차 라벨: 생활밀착지수 > 0 이면 생활밀착형(소매 우위가 구조적으로 의미 있는 경계), 아니면 한식/양식 비교로 B/C 분리

In [3]:
rule_df = compute_rule_scores(features)
print("1차 규칙기반 라벨 분포:")
print(rule_df['rule_segment'].value_counts())
rule_df[REGION_KEYS + ['생활밀착지수', '한식지향지수', '양식지향지수', 'rule_segment']].sort_values('생활밀착지수', ascending=False).head(10)


1차 규칙기반 라벨 분포:
rule_segment
생활밀착형      193
로컬미식형       59
프리미엄외식형      3
Name: count, dtype: int64


,SIDO_NM,CCG_NM,생활밀착지수,한식지향지수,양식지향지수,rule_segment
89,경상북도,고령군,66.876763,11.447305,2.914203,생활밀착형
84,경상남도,함안군,60.469072,13.024344,4.366596,생활밀착형
48,경기도,여주시,58.557046,12.774959,5.583591,생활밀착형
76,경상남도,창녕군,57.331328,12.248298,6.821620,생활밀착형
197,전라남도,신안군,56.678778,16.160267,4.780517,생활밀착형
67,경상남도,고성군,55.438164,15.245897,4.829454,생활밀착형
204,전라남도,진도군,55.185797,16.818263,3.251199,생활밀착형
95,경상북도,성주군,54.465859,14.618242,5.528650,생활밀착형
246,충청북도,음성군,53.383555,11.404702,8.315462,생활밀착형
200,전라남도,영암군,52.607088,11.278494,8.965216,생활밀착형


## 3-3. (B) 군집분석 — K-means

업종 비중 11차원 + 연령 비중 6차원(총 17차원)을 `StandardScaler`로 표준화한 뒤 K-means 수행.
K는 실루엣 스코어가 최대인 값을 채택한다 (K=3을 강제하지 않음, `random_state=42` 고정).

In [4]:
cluster_df, meta = run_kmeans_clustering(rule_df)

print("K별 실루엣 스코어:")
for k, s in meta['silhouette_by_k'].items():
    marker = " <- 최적" if k == meta['best_k'] else ""
    print(f"  K={k}: {s:.4f}{marker}")

print(f"\n최적 K = {meta['best_k']} (실루엣 스코어 = {meta['best_silhouette']:.4f})")


K별 실루엣 스코어:
  K=2: 0.1942
  K=3: 0.2060 <- 최적
  K=4: 0.1770
  K=5: 0.1841
  K=6: 0.1906
  K=7: 0.2026
  K=8: 0.1486

최적 K = 3 (실루엣 스코어 = 0.2060)


In [5]:
if meta['best_k'] != 3:
    print("⚠️ 경고: 실루엣 스코어상 최적 K가 3이 아닙니다 (§9-3 확인 필요 상황).")
    print("   이 노트북은 자동으로 '생활밀착형 1개 + 로컬미식형 1개 + 나머지 전부 프리미엄외식형' 규칙으로 대체했으나,")
    print("   본 주제(3대 세그먼트) 구성에 영향이 크다면 사용자에게 재확인이 필요합니다.")
else:
    print("[확인] 최적 K=3 — 실루엣 스코어 기준으로도 3대 세그먼트 구성이 자연스럽게 지지된다.")

print(f"\n군집별 크기: {cluster_df['cluster'].value_counts().sort_index().to_dict()}")


[확인] 최적 K=3 — 실루엣 스코어 기준으로도 3대 세그먼트 구성이 자연스럽게 지지된다.

군집별 크기: {0: 81, 1: 45, 2: 129}


### 3-3b. ⭐ K 선택 강건성 점검: 부트스트랩 군집 안정성(ARI)

위 실루엣 스코어를 보면 K=3(0.2060)이 최댓값이지만 K=7(0.2026)과의 격차가 0.0034에
불과해, 실루엣 스코어만으로는 "K=3이 유일하게 뚜렷한 최적값"이라 주장하기엔 근거가
얕다. 실루엣 스코어는 군집의 형태(조밀도·분리도)만 보므로, 독립적인 기준인
**서브샘플링 기반 안정성(Adjusted Rand Index)** 으로 K별 재현성을 추가 검증한다.

방법: 각 K에 대해 전체 데이터로 학습한 기준 군집 라벨을 만들고, 데이터의 80%를
무작위로 50회 반복 추출해 재학습한 뒤 기준 라벨과의 ARI를 측정한다. ARI가 높을수록
"표본이 조금 달라져도 같은 군집 구조가 재현된다"는 뜻이다.

In [6]:
from scoring import bootstrap_cluster_stability

stability_by_k = bootstrap_cluster_stability(meta['X_scaled'], k_range=range(2, 9), n_bootstrap=50)

print("K별 부트스트랩 안정성(평균 ARI, 높을수록 안정적):")
for k, s in stability_by_k.items():
    marker = " <- 실루엣 최적 K" if k == meta['best_k'] else ""
    print(f"  K={k}: ARI={s:.4f}{marker}")

best_stability_k = max(stability_by_k, key=stability_by_k.get)
print(f"\n안정성 기준 최적 K = {best_stability_k} (ARI={stability_by_k[best_stability_k]:.4f})")
if best_stability_k == meta['best_k']:
    print("[확인] 실루엣 스코어 기준 K와 안정성(ARI) 기준 K가 일치한다 — K=3 선택이 두 개의 "
          "독립적인 기준(형태 조밀도 + 재현 안정성)에서 교차검증됨.")
else:
    print(f"[주의] 실루엣 기준 K({meta['best_k']})와 안정성 기준 K({best_stability_k})가 다르다 — "
          "두 기준이 서로 다른 K를 지지하므로 K 선택 근거를 재검토하고 보고서에 이 불일치를 명시해야 한다.")

K별 부트스트랩 안정성(평균 ARI, 높을수록 안정적):
  K=2: ARI=0.9154
  K=3: ARI=0.9053 <- 실루엣 최적 K
  K=4: ARI=0.5250
  K=5: ARI=0.7026
  K=6: ARI=0.5938
  K=7: ARI=0.6058
  K=8: ARI=0.6215

안정성 기준 최적 K = 2 (ARI=0.9154)
[주의] 실루엣 기준 K(3)와 안정성 기준 K(2)가 다르다 — 두 기준이 서로 다른 K를 지지하므로 K 선택 근거를 재검토하고 보고서에 이 불일치를 명시해야 한다.


### 군집별 프로파일 확인 (군집 -> 세그먼트 라벨 매핑 근거)

In [7]:
profile_cols = ['생활밀착지수', '한식지향지수', '양식지향지수', '건당평균결제금액', '총건수']
print(cluster_df.groupby('cluster')[profile_cols].mean().round(2))

cluster_to_seg = map_clusters_to_segments(cluster_df)
print("\n군집 -> 세그먼트 매핑:", cluster_to_seg)


         생활밀착지수  한식지향지수  양식지향지수  건당평균결제금액        총건수
cluster                                             
0         36.69   21.05    7.73  18276.96  289464.25
1          1.20   27.00   17.74  14640.12  361443.31
2          6.70   32.07   10.04  18402.23  277505.22

군집 -> 세그먼트 매핑: {np.int32(0): '생활밀착형', np.int32(2): '로컬미식형', 1: '프리미엄외식형'}


## 3-4. (C) 상호 검증: 규칙기반 vs 군집분석

두 방법의 라벨 일치도를 crosstab으로 확인한다. 완전히 일치할 필요는 없다 —
규칙기반은 업종 비중 2개 지수만 보는 단순 휴리스틱이고, 군집분석은 17차원 전체(업종+연령)를
종합적으로 보기 때문에 차이가 나는 것 자체가 군집분석의 부가가치를 보여준다.

In [8]:
final_df, final_meta = classify_segments(features)

print(f"규칙기반 vs 군집분석 일치도: {final_meta['agreement_rate']*100:.1f}%")
print("\nCrosstab (행=규칙기반, 열=군집분석 기반 최종세그먼트):")
print(final_meta['crosstab'])
print("\n[해석] 일치도가 100%가 아닌 것은 정상이다. 규칙기반은 업종 비중 2개 지수만 보는 단순 지표이고,")
print("군집분석은 업종 11종 + 연령 6종을 종합적으로 반영하므로 더 정교하게 갈린다.")
print("최종 세그먼트는 군집분석(B) 결과를 채택하고, 규칙기반(A)은 교차검증용 참고 지표로 유지한다.")


규칙기반 vs 군집분석 일치도: 48.6%

Crosstab (행=규칙기반, 열=군집분석 기반 최종세그먼트):
segment       로컬미식형  생활밀착형  프리미엄외식형
rule_segment                       
로컬미식형            42      2       15
생활밀착형            87     79       27
프리미엄외식형           0      0        3

[해석] 일치도가 100%가 아닌 것은 정상이다. 규칙기반은 업종 비중 2개 지수만 보는 단순 지표이고,
군집분석은 업종 11종 + 연령 6종을 종합적으로 반영하므로 더 정교하게 갈린다.
최종 세그먼트는 군집분석(B) 결과를 채택하고, 규칙기반(A)은 교차검증용 참고 지표로 유지한다.


In [9]:
# 안정성 기준으로는 K=2가 K=3보다 근소하게 앞선다(0.9154 vs 0.9053). 이 차이가
# "K=3 대신 K=2를 써야 한다"는 뜻인지 확인하기 위해, K=2 군집이 실제로 무엇을
# 나누는지 K=3 최종 세그먼트와 교차표로 대조한다.
from sklearn.cluster import KMeans

km2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit(meta['X_scaled'])
final_df['cluster_k2'] = km2.labels_
k2_crosstab = pd.crosstab(final_df['segment'], final_df['cluster_k2'])
print("K=2 군집 vs K=3 최종 세그먼트 교차표:")
print(k2_crosstab)
print()
print("[해석] K=2는 생활밀착형(81곳 중 80곳)과 로컬미식형(129곳 중 128곳)은 각각 거의 "
      "그대로 한 군집에 담아내지만, 프리미엄외식형(45곳)은 두 군집에 27:18로 흩어져 "
      "어느 쪽에도 깔끔하게 속하지 못한다. 즉 K=2는 '프리미엄외식형'이라는 세 번째 성격을 "
      "구조적으로 표현하지 못하며, 이는 04번 노트북에서 외부(법무부) 데이터로 별도 검증된 "
      "유학계열 비중 차이(41.6% vs 12.7%, p=1.70×10⁻¹⁶)를 카드데이터만의 군집 구조로는 K=2가 "
      "포착하지 못한다는 뜻이다. 따라서 안정성 지표상 근소한 차이에도 불구하고, 실루엣 "
      "스코어 최적값이면서 안정성도 K=2에 버금가게 높고(0.905) 세 성격을 모두 보존하는 K=3을 "
      "최종 채택한다 — 이는 임의의 해석이 아니라 4번 노트북의 외부 데이터 교차검증으로 "
      "뒷받침되는 결정이다.")

K=2 군집 vs K=3 최종 세그먼트 교차표:
cluster_k2   0    1
segment            
로컬미식형        1  128
생활밀착형       80    1
프리미엄외식형     27   18

[해석] K=2는 생활밀착형(81곳 중 80곳)과 로컬미식형(129곳 중 128곳)은 각각 거의 그대로 한 군집에 담아내지만, 프리미엄외식형(45곳)은 두 군집에 27:18로 흩어져 어느 쪽에도 깔끔하게 속하지 못한다. 즉 K=2는 '프리미엄외식형'이라는 세 번째 성격을 구조적으로 표현하지 못하며, 이는 04번 노트북에서 외부(법무부) 데이터로 별도 검증된 유학계열 비중 차이(41.6% vs 12.7%, p=1.70×10⁻¹⁶)를 카드데이터만의 군집 구조로는 K=2가 포착하지 못한다는 뜻이다. 따라서 안정성 지표상 근소한 차이에도 불구하고, 실루엣 스코어 최적값이면서 안정성도 K=2에 버금가게 높고(0.905) 세 성격을 모두 보존하는 K=3을 최종 채택한다 — 이는 임의의 해석이 아니라 4번 노트북의 외부 데이터 교차검증으로 뒷받침되는 결정이다.


## 3-5. 최종 세그먼트 분포 및 §3-2 예시 지역 재확인

In [10]:
print("최종 세그먼트 분포:")
print(final_df['segment'].value_counts())
print()
print(final_df.groupby('segment')[['건당평균결제금액', '총소비금액', '총건수']].mean().round(0))


최종 세그먼트 분포:
segment
로컬미식형      129
생활밀착형       81
프리미엄외식형     45
Name: count, dtype: int64

         건당평균결제금액         총소비금액       총건수
segment                                  
로컬미식형     18402.0  5.011217e+09  277505.0
생활밀착형     18277.0  4.901278e+09  289464.0
프리미엄외식형   14640.0  5.214201e+09  361443.0


In [11]:
# §3-2에서 사전탐색 예시로 언급된 지역들이 전수 재계산 후에도 같은 세그먼트로 분류되는지 참고용으로 확인
# (이 리스트를 분류 로직에 하드코딩하지 않았음 — 순수하게 위 알고리즘 결과를 사후 대조만 함)
example_regions = {
    '시흥시': 'A(생활밀착형)', '안산시 단원구': 'A(생활밀착형)', '화성시 만세구': 'A(생활밀착형)',
    '평택시': 'A(생활밀착형)', '아산시': 'A(생활밀착형)', '김해시': 'A(생활밀착형)',
    '광산구': 'A(생활밀착형)', '달서구': 'A(생활밀착형)', '김포시': 'A(생활밀착형)',
    '제주시': 'B(로컬미식형)', '서귀포시': 'B(로컬미식형)',
    '강남구': 'C(프리미엄외식형)', '연수구': 'C(프리미엄외식형)',
}
check = final_df[final_df['CCG_NM'].isin(example_regions)][['SIDO_NM', 'CCG_NM', 'segment']].copy()
check['사전탐색_예상'] = check['CCG_NM'].map(example_regions)
print(check.to_string(index=False))


SIDO_NM  CCG_NM segment    사전탐색_예상
    경기도     김포시   생활밀착형   A(생활밀착형)
    경기도     시흥시   로컬미식형   A(생활밀착형)
    경기도 안산시 단원구   로컬미식형   A(생활밀착형)
    경기도     평택시   생활밀착형   A(생활밀착형)
    경기도 화성시 만세구   생활밀착형   A(생활밀착형)
   경상남도     김해시   생활밀착형   A(생활밀착형)
  광주광역시     광산구   생활밀착형   A(생활밀착형)
  대구광역시     달서구   생활밀착형   A(생활밀착형)
  서울특별시     강남구 프리미엄외식형 C(프리미엄외식형)
  인천광역시     연수구 프리미엄외식형 C(프리미엄외식형)
제주특별자치도    서귀포시   로컬미식형   B(로컬미식형)
제주특별자치도     제주시   로컬미식형   B(로컬미식형)
   충청남도     아산시   생활밀착형   A(생활밀착형)


## 3-6. 산출물 저장: `results/segment_classification.csv`

In [12]:
out_cols = REGION_KEYS + [
    'segment', 'rule_segment', 'cluster',
    '생활밀착지수', '한식지향지수', '양식지향지수',
    '건당평균결제금액', '월별변동계수CV', '총소비금액', '총건수', 'low_sample',
]
result = final_df[out_cols].copy()

from pathlib import Path
out_path = Path('../results/segment_classification.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"[OK] 저장: {out_path} ({result.shape[0]}행 x {result.shape[1]}열)")
result.head()


[OK] 저장: ../results/segment_classification.csv (255행 x 13열)


,SIDO_NM,CCG_NM,segment,rule_segment,cluster,생활밀착지수,한식지향지수,양식지향지수,건당평균결제금액,월별변동계수CV,총소비금액,총건수,low_sample
0,강원특별자치도,강릉시,프리미엄외식형,로컬미식형,1,-1.070947,30.266999,17.052776,18695.089002,0.084337,4200020000,224659,False
1,강원특별자치도,고성군,프리미엄외식형,생활밀착형,1,10.808558,17.261794,23.921946,10296.345613,0.260488,695930000,67590,False
2,강원특별자치도,동해시,로컬미식형,생활밀착형,2,25.253868,23.440077,10.794351,21761.658816,0.067390,1436770000,66023,False
3,강원특별자치도,삼척시,로컬미식형,로컬미식형,2,-2.560137,34.777321,12.668549,21388.014644,0.166590,665980000,31138,False
4,강원특별자치도,속초시,로컬미식형,로컬미식형,2,-6.873140,36.405768,13.833582,20911.231761,0.109960,3081270000,147350,False


## 요약

- K-means 실루엣 스코어 기준 최적 K=3으로, 3대 세그먼트 구성이 데이터로 뒷받침됨.
- K=3과 K=7의 실루엣 스코어 격차(0.0034)가 근소해 부트스트랩 안정성(ARI)으로 교차검증한 결과,
  K=3(0.905)이 K=7(0.606)보다 훨씬 재현 안정적임을 확인. 안정성 1위인 K=2(0.915)는 프리미엄외식형을
  두 군집에 27:18로 흩어버려 세 번째 성격을 표현하지 못해 최종 채택하지 않음.
- 규칙기반과 군집분석의 일치도는 부분적이며, 이는 군집분석이 더 많은 차원을 반영하기 때문으로 해석.
- §3-2 예시 지역 중 로컬미식형(B)·프리미엄외식형(C) 후보는 전수 재계산 후에도 전부 동일하게 분류됨.
  생활밀착형(A) 후보 9곳 중 2곳(시흥시, 안산시 단원구)은 재계산 결과 로컬미식형으로 이동 — 예시치가
  일부 지역만 표본으로 삼은 값이었음을 보여주는 사례.

다음 단계(`04_external_validation.ipynb`)에서 법무부 체류자격 데이터로 이 세그먼트 분류를 검증한다.